In [ ]:
import json

import polars as pl
import requests

def load_club_data(file):
    """
    Loads club data from a CSV file.

    Arguments:
    - file (str): The file path of the CSV file.

    Returns:
    - data (pl.DataFrame): The loaded club data as a pandas DataFrame.
    """
    data = pl.read_csv(file)
    return data

data = load_club_data("postcodes_master_2025.csv")

In [2]:
def get_distance(point1: dict, point2: dict) -> tuple:
    """
    Gets the driving distance and duration between two points using
    http://project-osrm.org/docs/v5.10.0/api/#nearest-service

    Arguments:
    - point1 (dict): A dictionary representing the latitude and longitude of the first point.
    - point2 (dict): A dictionary representing the latitude and longitude of the second point.

    Returns:
    - tuple: A tuple containing the distance (in meters) and duration (in seconds) of the route.
    """
    url = (
            "http://router.project-osrm.org/route/v1/driving/"
            f"{point1['longitude']},{point1['latitude']};"
            f"{point2['longitude']},{point2['latitude']}"
            "?overview=false&alternatives=false"
        )
    r = requests.get(url)

    # get the distance from the returned values
    route = json.loads(r.content)["routes"][0]
    return (route["distance"], route["duration"])

In [ ]:
def create_dist_array(step: str):
    """
    Creates an array of distances between all combinations of points within the same league.
    Takes a while due to calling the API (bottleneck)

    Arguments:
    - step (str): The league step to filter the dataset by (e.g. 'Step 3', 'professional').

    Returns:
    - dist_array (list): A list of tuples containing the origin index, destination index,
    duration (in seconds), and distance (in meters) between each pair of points.
    """
    dist_array = []

    # Filter the dataset by the given step
    step_data = data.filter(pl.col("step") == step)
    step_data = step_data.rename({col: str(col).strip() for col in step_data.columns})

    for r in step_data.iter_rows(named=True):
        point1 = {"latitude": r["latitude"], "longitude": r["longitude"]}
        team1 = r["team"]
        league = r["league"]

        same_league_df = step_data.filter(pl.col("league") == league)

        for o in same_league_df.iter_rows(named=True):
            team2 = o["team"]

            if team1 == team2:
                continue

            point2 = {"latitude": o["latitude"], "longitude": o["longitude"]}
            dist, duration = get_distance(point1, point2)
            dist_array.append((team1, team2, duration, dist))
    
    return dist_array

dist_array = create_dist_array("Step 1")

In [ ]:
def create_distances_df():
    """
    Creates a Polars DataFrame of distances between all combinations of points.

    Returns:
    - distances_df (pl.DataFrame): The DataFrame containing the distances between each pair of points,
    including origin and destination names, distance in miles, duration in minutes, and a fixture key.
    """

    distances_df = pl.DataFrame(dist_array, schema=["origin", "destination", "duration(s)", "distance(m)"])

    distances_df = distances_df.join(
    data.select([pl.col("team").alias("origin_name"), pl.col("league").alias("origin_league")])
        .with_columns(pl.Series("origin", data.get_column("team"))),
    on="origin",
    how="left"
    )

    distances_df = distances_df.join(
    data.select([pl.col("team").alias("destination_name"), pl.col("league").alias("destination_league")])
        .with_columns(pl.Series("destination", data.get_column("team"))),
    on="destination",
    how="left"
    )

    distances_df = distances_df.filter(
    pl.col("origin_league") == pl.col("destination_league")
    )
    
    distances_df = distances_df.with_columns(
    (pl.col("distance(m)") * 0.000621371)
    .round(1)
    .alias("distance(miles)")
    )

    distances_df = distances_df.with_columns(
        (pl.col("duration(s)") / 60).round(1).alias("duration(min)")
    )

    distances_df = distances_df.with_columns(
    (pl.col("destination_name").str.strip_chars() + "-" + pl.col("origin_name").str.strip_chars())
    .alias("fixture_key")
    )

    distances_df = distances_df.drop("origin_name","destination_name","destination_league")

    return distances_df

In [16]:
journeys_df = create_distances_df()
journeys_df

origin,destination,duration(s),distance(m),origin_league,distance(miles),duration(min),fixture_key
str,str,f64,f64,str,f64,f64,str
"""Aldershot Town""","""Altrincham""",13991.7,324543.4,"""National League""",201.7,233.2,"""Altrincham-Aldershot Town"""
"""Aldershot Town""","""Boreham Wood""",3993.7,84088.7,"""National League""",52.3,66.6,"""Boreham Wood-Aldershot Town"""
"""Aldershot Town""","""Boston United""",10487.2,241095.0,"""National League""",149.8,174.8,"""Boston United-Aldershot Town"""
"""Aldershot Town""","""Brackley Town""",5695.9,122125.9,"""National League""",75.9,94.9,"""Brackley Town-Aldershot Town"""
"""Aldershot Town""","""Braintree Town""",7025.5,166250.0,"""National League""",103.3,117.1,"""Braintree Town-Aldershot Town"""
…,…,…,…,…,…,…,…
"""Yeovil Town""","""Sutton United""",9428.8,211149.3,"""National League""",131.2,157.1,"""Sutton United-Yeovil Town"""
"""Yeovil Town""","""Tamworth""",11562.4,262744.0,"""National League""",163.3,192.7,"""Tamworth-Yeovil Town"""
"""Yeovil Town""","""Truro City""",6567.8,144540.0,"""National League""",89.8,109.5,"""Truro City-Yeovil Town"""
